In [1]:
# -*- coding: utf-8 -*-
import os
os.environ.setdefault("OMP_NUM_THREADS", "12")
os.environ.setdefault("MKL_NUM_THREADS", "12")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import platform
import re
import warnings
import unicodedata
from datetime import datetime
from typing import List, Tuple
import time
import math

import torch
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
transformers.logging.set_verbosity_info()

from docx import Document
from striprtf.striprtf import rtf_to_text

try:
    import polars as pl
    HAS_POLARS = True
except Exception:
    HAS_POLARS = False

# ------------------------------
# RAW CONFIGURATION
# ------------------------------
MODE = 'title'   # 'title' alebo 'keyword'

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'

THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
STOPWORDS_TXT  = './Input/stopword_dictionary.txt'

LLM_PRIMARY  = 'marcuscedricridia/8B-Nemotaur-IT'
LLM_FALLBACK = 'nvidia/Llama-3.1-Nemotron-Nano-4B-v1.1'

# Inference / batching
MAX_NEW_TOKENS   = 128     
BATCH_SIZE       = 4         
DO_SAMPLE        = True
TEMPERATURE      = 0.6
TOP_P            = 0.95
REPETITION_PENALTY = 2.0
MAX_TEXT_CHARS   = 8000

REASONING_FOR_PRIMARY  = True
REASONING_FOR_FALLBACK = True
REASONING_SAMPLES = 2         
REASONING_TEMP    = 0.6
REASONING_TOP_P   = 0.95


MAX_INPUT_TOKENS  = 1400      

PRI_MAX  = 15                
SAMP_MAX = 25               

# Threading (PyTorch)
torch.set_num_threads(12)

# CUDA mem housekeeping
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


# ------------------------------
# CUDA allocator and flash off
# ------------------------------
def configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

configure_cuda_allocator()
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")


# ------------------------------
# UTIL: stopwords, diacritics
# ------------------------------
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f'Loaded {len(words)} stopwords from {path}')
        return words
    except FileNotFoundError:
        print(f'No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'Failed to load stopwords: {e}')
        return set()

STOPWORDS = load_stopwords()

def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))


# ------------------------------
# TEZAURUS loader (Parquet/CSV/Excel)
# ------------------------------
def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)

    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))

    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]

    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    pq_path = os.path.splitext(xlsx_path)[0] + ".parquet"
    csv_path = os.path.splitext(xlsx_path)[0] + ".csv"

    if HAS_POLARS and os.path.exists(pq_path):
        df = pl.read_parquet(pq_path)
        if column not in df.columns:
            raise ValueError(f'Column "{column}" not in {pq_path}. Available: {df.columns}')
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list()
    elif HAS_POLARS and os.path.exists(csv_path):
        df = pl.read_csv(csv_path)
        if column not in df.columns:
            raise ValueError(f'Column "{column}" not in {csv_path}. Available: {df.columns}')
        raw = df[column].drop_nulls().cast(pl.Utf8).to_list()
    else:
        if not os.path.exists(xlsx_path):
            raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
        df = pd.read_excel(xlsx_path, engine='openpyxl')
        if column not in df.columns:
            raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
        raw = df[column].dropna().astype(str).tolist()

    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', str(cell)))

    seen, terms = set(), []
    for t in parts:
        t = (t or "").strip()
        if not t:
            continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)

    print(f'Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    hits = {}
    if not text:
        return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]


# ------------------------------
# DOCX / RTF splitting
# ------------------------------
_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []


# ------------------------------
# LLM load (VRAM-friendly)
# ------------------------------
def _post_model_low_vram_config(model, tok):
    # Menšia VRAM počas decode:
    model.config.use_cache = False
    # Eager attention (menej VRAM než sdpa/flash)
    setattr(model.config, "_attn_implementation", "eager")

    # Fixy pre decoder-only:
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id

def load_primary_llm():
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,   # šetrné k VRAM
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        LLM_PRIMARY,
        quantization_config=qconf,
        device_map=device_map,
        dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_PRIMARY)
    _post_model_low_vram_config(model, tok)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        device_map=device_map,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        return_full_text=False,
        pad_token_id=tok.pad_token_id,
    )
    return pipe, f"primary:{LLM_PRIMARY}"

def load_fallback_llm():
    for k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_FLASH_ATTENTION"):
        if k in os.environ:
            os.environ.pop(k)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        LLM_FALLBACK,
        device_map=device_map,
        dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    tok = AutoTokenizer.from_pretrained(LLM_FALLBACK)
    _post_model_low_vram_config(model, tok)

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        device_map=device_map,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        repetition_penalty=REPETITION_PENALTY,
        return_full_text=False,
        pad_token_id=tok.pad_token_id,
    )
    return pipe, f"fallback:{LLM_FALLBACK}"

try:
    LLM_PIPE, USED_MODEL = load_primary_llm()
    print(f"[LLM] Loaded {USED_MODEL}")
except Exception as e:
    print(f"[WARN] Primary model failed: {e}")
    LLM_PIPE, USED_MODEL = load_fallback_llm()
    print(f"[LLM] Loaded {USED_MODEL}")


# ------------------------------
# Prompt helper + clamp
# ------------------------------
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def clamp_prompt_tokens(text: str, tok, max_input_tokens: int = MAX_INPUT_TOKENS) -> str:
    ids = tok(text, return_tensors=None, add_special_tokens=False).input_ids
    if len(ids) <= max_input_tokens:
        return text
    keep = ids[-max_input_tokens:]
    return tok.decode(keep, skip_special_tokens=True)

def prompt_title(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = "\n".join(th_priority[:PRI_MAX]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:SAMP_MAX])
    return (
        "ÚLOHA: Navrhni JEDEN presný a zrozumiteľný právny nadpis (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecný, bez bodky na konci, žiadne úvodzovky. Preferuj slovenčinu.\n"
        "Vráť len samotný nadpis, nič iné.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "NADPIS:"
    )

def prompt_keyword(text: str, th_priority: List[str], th_sample: List[str]) -> str:
    pri = ", ".join(th_priority[:PRI_MAX]) if th_priority else "(žiadne priame zhody)"
    samp = ", ".join(th_sample[:SAMP_MAX])
    return (
        "ÚLOHA: Vyber JEDEN najvhodnejší právny pojem (1–4 slová) k nasledujúcemu textu."
        " Preferuj pojmy v slovenčine. Vráť len samotný pojem, bez vysvetlenia a bez bodky.\n\n"
        f"TEZAURUS – PRIO: {pri}\n"
        f"TEZAURUS – VÝBER: {samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "POJEM:"
    )


# ------------------------------
# Batch call
# ------------------------------
def safe_llm_call_batch(
    pipe,
    inputs,                      
    batch_size: int,
    num_return_sequences: int = 1,
    max_new_tokens: int = None,
    temperature: float = None,
    top_p: float = None,
    max_retries: int = 4,
    min_batch: int = 1,
    min_new_tokens: int = 32,
    min_samples: int = 1,
    sleep_s: float = 0.2,
):
    """
    Vráti trojicu: (outputs, used_batch_size, used_num_return_sequences)
    kde outputs je buď List[Dict] alebo List[List[Dict]] podľa HF pipeline.
    """
    bs   = max(batch_size, 1)
    nrs  = max(num_return_sequences, 1)
    mnt  = max_new_tokens if max_new_tokens is not None else None
    tries = 0

    while True:
        try:
            kwargs = dict(batch_size=bs, num_return_sequences=nrs)
            if mnt is not None: kwargs["max_new_tokens"] = mnt
            if temperature is not None: kwargs["temperature"] = temperature
            if top_p is not None: kwargs["top_p"] = top_p
            out = pipe(inputs, **kwargs)
            return out, bs, nrs  # ← vraciame aj realne použité hodnoty
        except RuntimeError as e:
            msg = str(e).lower()
            if "out of memory" not in msg and "cuda out of memory" not in msg:
                raise
            tries += 1
            if tries > max_retries and bs <= min_batch and nrs <= min_samples and (mnt is None or mnt <= min_new_tokens):
                raise

            if bs > min_batch:
                bs = max(min_batch, bs // 2)
            elif nrs > min_samples:
                nrs = max(min_samples, nrs - 1)
            elif mnt is None or mnt > min_new_tokens:
                mnt = max(min_new_tokens, (mnt or 256) // 2)

            torch.cuda.empty_cache()
            time.sleep(sleep_s)



# ------------------------------
# Batch generators
# ------------------------------
def _normalize_output_item(item) -> str:
    """Z itemu vráť prvú čistú riadkovú odpoveď."""
    rec = item[0] if isinstance(item, list) else item
    raw = (
        rec.get("generated_text") if isinstance(rec, dict) else None
    ) or (
        rec.get("summary_text") if isinstance(rec, dict) else None
    ) or str(rec)

    txt = re.sub(r'^[\-\•\:\s]+', '', raw).strip()
    txt = re.sub(r'[\.，。…]+$', '', txt).strip()
    txt = txt.strip(' "„”')
    return txt.splitlines()[0].strip()

def batched_generate_texts(prompts: List[str], batch_size: int = BATCH_SIZE) -> List[str]:
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        out = safe_llm_call_batch(
            LLM_PIPE,
            batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=1,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE if DO_SAMPLE else None,
            top_p=TOP_P if DO_SAMPLE else None,
        )
        for item in out:
            outputs.append(_normalize_output_item(item))
    if len(outputs) != len(prompts):
        print(f"[WARN] batched_generate_texts: outputs={len(outputs)} != inputs={len(prompts)}")
    return outputs

def batched_generate_texts_reasoned(jobs, batch_size: int = 2) -> List[str]:
    """
    Robustné grupovanie výstupov:
    - ak HF vráti List[List[Dict]] → per-input máme priamo zoznam kandidátov,
    - ak HF vráti plochý List[Dict] → použijeme skutočný used_nrs (z safe_llm_call_batch),
        a poslednú skupinu osekáme podľa zvyšného počtu itemov.
    """
    prompts = []
    for (_, _, _, _, p) in jobs:
        p = clamp_prompt_tokens(p, LLM_PIPE.tokenizer, MAX_INPUT_TOKENS)
        prompts.append(p)

    final_out = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]

        out, used_bs, used_nrs = safe_llm_call_batch(
            LLM_PIPE,
            batch,
            batch_size=min(batch_size, len(batch)),
            num_return_sequences=max(1, min(REASONING_SAMPLES, 6)),
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=REASONING_TEMP,
            top_p=REASONING_TOP_P,
        )

        def norm(x):
            return _normalize_output_item(x)

        if isinstance(out, list) and out and isinstance(out[0], list):
            for per_input in out:
                cands = [norm(item) for item in per_input]
                # scoring
                scored = []
                for t in cands:
                    words = len(t.split())
                    length_score = 1.0 if 3 <= words <= 12 else 0.4
                    sk_bonus = 0.2 if re.search(r"[áäčďéíľĺňóôŕšťúýžÁÄČĎÉÍĽĹŇÓÔŔŠŤÚÝŽ]", t) else 0.0
                    dot_penalty = -0.2 if t.endswith('.') else 0.0
                    scored.append((length_score + sk_bonus + dot_penalty, t))
                best = max(scored, key=lambda x: x[0])[1] if scored else ""
                final_out.append(best)
        else:
            flat = out if isinstance(out, list) else []
            cursor = 0
            for bix in range(len(batch)):
                remaining = len(flat) - cursor
                take = min(used_nrs, remaining) if used_nrs > 0 else remaining
                if take <= 0:
                    final_out.append("")
                    continue
                slice_items = flat[cursor:cursor + take]
                cursor += take

                cands = [norm(item) for item in slice_items]
                scored = []
                for t in cands:
                    words = len(t.split())
                    length_score = 1.0 if 3 <= words <= 12 else 0.4
                    sk_bonus = 0.2 if re.search(r"[áäčďéíľĺňóôŕšťúýžÁÄČĎÉÍĽĹŇÓÔŔŠŤÚÝŽ]", t) else 0.0
                    dot_penalty = -0.2 if t.endswith('.') else 0.0
                    scored.append((length_score + sk_bonus + dot_penalty, t))
                best = max(scored, key=lambda x: x[0])[1] if scored else ""
                final_out.append(best)

    if len(final_out) != len(prompts):
        print(f"[WARN] reasoned: outputs={len(final_out)} != inputs={len(prompts)} (často OK po OOM downscale)")

    return final_out

# ------------------------------
# DRIVER
# ------------------------------
def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    # 1) Naplánuj joby a priprav prompty dopredu
    jobs = []   # (mode, file, header, prio_terms, prompt)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue

        if not secs:
            print(f'No text in {f}')
            continue

        for header, text in secs:
            if not text.strip():
                continue

            short = truncate_text(text, MAX_TEXT_CHARS)
            prio_terms = terms_matched_in_text(text, max_terms=30)
            sample_terms = prio_terms if prio_terms else CANON_TERMS[:SAMP_MAX]

            if MODE == 'title':
                prompt = prompt_title(short, prio_terms, sample_terms)
            else:
                prompt = prompt_keyword(short, prio_terms, sample_terms)

            # clamp na tokeny – pred batchom
            prompt = clamp_prompt_tokens(prompt, LLM_PIPE.tokenizer, MAX_INPUT_TOKENS)
            jobs.append((MODE, f, header, prio_terms, prompt))

        print(f'Queued {f} with {len(secs)} sections.')

    if not jobs:
        print('No jobs queued.')
        return pd.DataFrame([])

    # 2) Batch LLM inference (s reasoningom ak zapnutý)
    reasoning_active = (
        (USED_MODEL.startswith("primary:")  and REASONING_FOR_PRIMARY) or
        (USED_MODEL.startswith("fallback:") and REASONING_FOR_FALLBACK)
    )

    if reasoning_active and REASONING_SAMPLES > 1:
        gens = batched_generate_texts_reasoned(jobs, batch_size=2)
    else:
        prompts = [j[-1] for j in jobs]
        gens = batched_generate_texts(prompts, batch_size=BATCH_SIZE)

    # 3) Zloženie výsledkov
    for (mode, f, header, prio_terms, _), gen in zip(jobs, gens):
        if mode == 'title':
            rows.append({
                "file": f,
                "header": header,
                "suggested_title": gen,
                "method": f"llm-only ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })
        else:
            rows.append({
                "file": f,
                "header": header,
                "top_keyword": gen,
                "method": f"llm-only ({USED_MODEL})",
                "priority_terms": "; ".join(prio_terms[:20])
            })

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == 'title':
        df = pd.DataFrame(rows, columns=["file","header","suggested_title","method","priority_terms"])
        stub = "llm_only_titles"
    else:
        df = pd.DataFrame(rows, columns=["file","header","top_keyword","method","priority_terms"])
        stub = "llm_only_keywords"

    csv_path  = os.path.join(OUTPUT_DIR, f"{stub}_{today}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved: {csv_path}")

    return df

# ------------------------------
# MAIN
# ------------------------------
if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))


Loaded 76 stopwords from ./Input/stopword_dictionary.txt
Thesaurus terms loaded: 3000


loading configuration file config.json from cache at C:\Users\nyx3t\.cache\huggingface\hub\models--marcuscedricridia--8B-Nemotaur-IT\snapshots\78f51fcf89adbf656532cc9842032350c95e46b7\config.json
Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing LlamaForCausalLM.

All the weights of LlamaForCausalLM were initialized from the model checkpoint at marcuscedricridia/8B-Nemotaur-IT.
If your task is similar to the task the model of the checkpoint was trained on, you can already use LlamaForCausalLM for predictions without further training.
Generation config file not found, using a generation config created from the model config.
loading file tokenizer.json from cache at C:\Users\nyx3t\.cache\huggingface\hub\models--marcuscedricridia--8B-Nemotaur-IT\snapshots\78f51fcf89adbf656532cc9842032350c95e46b7\tokenizer.json
loading file tokenizer.model from cache at None
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at C:\Users\nyx3t\.cache\huggingface\hub\models--marcuscedricridia--8B-Nemotaur-IT\snapshots\78f51fcf89adbf656532cc9842032350c95e46b7\special_tokens_map.json
loading file tokenizer_config.json from cache at C:\Users\nyx3t\.c

[LLM] Loaded primary:marcuscedricridia/8B-Nemotaur-IT
Queued 117694.docx with 8 sections.
Queued 117695.docx with 2 sections.
Queued 117696.docx with 11 sections.
Queued 117697.docx with 15 sections.
Queued 117698.docx with 8 sections.
Queued 117699.docx with 18 sections.
Queued 117700.docx with 1 sections.
Queued 117701.docx with 6 sections.
Queued 117702.docx with 12 sections.
Queued 117703.docx with 10 sections.


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Saved: ./Output\llm_only_titles_2025-11-03.csv


,file,header,suggested_title,method,priority_terms
0,117694.docx,__DOCUMENT__,Dispozície nad bytovým socialistickým majetcom,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; vyporiadanie; majetok (imanie); majeto...
1,117694.docx,po-heading-id_EHU0iUd9Y0yOqqj7VGbM1A,Je rozhodnutelnou otázanou otázana legálnosť v...,llm-only (primary:marcuscedricridia/8B-Nemotau...,podiel; vyporiadanie; spoločník; autor
2,117694.docx,po-heading-id_c6ihO-fo_kmbMqy9OwPZ5g,Uznesení Najvyších sudcov Slovenskeho republik...,llm-only (primary:marcuscedricridia/8B-Nemotau...,uznesenie
3,117694.docx,po-heading-id_aWeISBCNy0mMqxGjUkaTNw,"Otazkovacia formulacke predmet ""Opravneneho"" D...",llm-only (primary:marcuscedricridia/8B-Nemotau...,otázka
4,117694.docx,po-heading-id_VtH6pbc6j0qtnD6ytAcu1w,Záruka spravovaním práva - problématické aspek...,llm-only (primary:marcuscedricridia/8B-Nemotau...,konanie (štádium procesného konania); majetok ...
5,117694.docx,po-heading-id_aWuBOYjuME-jlXoxHWEcHg,Neexistencia právnicích závazků vyplývającích ...,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; právo; rozsudok; skutočnosti; dediči; ...
6,117694.docx,po-heading-id_-ctrw_j88Uazc2-86wvAAQ,Úloha ochrone privatenostės vo vnútorném správ...,llm-only (primary:marcuscedricridia/8B-Nemotau...,dohoda; majetok (imanie); vyporiadanie; majeto...
7,117694.docx,po-heading-id_kizOD8s8zEyXvfzM_OhAwg,SLOVENSKÝ ZÁKON - KOMENTOVANIE A PRAX,llm-only (primary:marcuscedricridia/8B-Nemotau...,zmier; vyporiadanie; uznesenie; majetok (imani...
8,117695.docx,__DOCUMENT__,Spravnaním právnych norm,llm-only (primary:marcuscedricridia/8B-Nemotau...,bremeno; skutočnosti; norma; dôkaz; služba (st...
9,117695.docx,po-heading-id_1OMtQ1Gs9kWEadIcZhz9ig,Spravnaním právu - nový trend?,llm-only (primary:marcuscedricridia/8B-Nemotau...,bremeno; skutočnosti; norma; dôkaz; služba (st...
